In [1]:
import boto3
import pandas as pd
from botocore.config import Config

S3_ENDPOINT = 'https://seaweedfs:8333'
S3_REGION = 'us-east-1'
S3_CA_BUNDLE = '/etc/seaweedfs/tls/ca.crt'

s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id='admin',
    aws_secret_access_key='secret2',
    region_name=S3_REGION,
    verify=S3_CA_BUNDLE,
    config=Config(s3={'addressing_style': 'path'}),
)

In [2]:
bucket_response = s3.list_buckets()

buckets = pd.DataFrame(
    [
        {
            'Name': bucket['Name'],
            'CreationDate': bucket['CreationDate'],
        }
        for bucket in bucket_response.get('Buckets', [])
    ],
    columns=['Name', 'CreationDate'],
)

display(buckets)

,Name,CreationDate
0,delta-lakehouse,2026-09-11 16:22:02+00:00
1,trino-lakehouse,2026-09-09 12:27:26+00:00


In [3]:
bucket_name = 'trino-lakehouse'
objects = []

for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket_name):
    objects.extend(
        {
            'Key': item['Key'],
            'Size': item['Size'],
            'LastModified': item['LastModified'],
        }
        for item in page.get('Contents', [])
    )

bucket_objects = pd.DataFrame(
    objects,
    columns=['Key', 'Size', 'LastModified'],
).sort_values('Key', ignore_index=True)

display(bucket_objects)

,Key,Size,LastModified
0,nyc_gov.db/,0,2026-09-09 14:54:27+00:00
1,nyc_gov.db/country_codes-9a177fe4c3ba4674ba94c...,2656,2026-09-11 20:48:35+00:00
2,nyc_gov.db/country_codes-9a177fe4c3ba4674ba94c...,1688,2026-09-11 20:48:35+00:00
3,nyc_gov.db/country_codes-9a177fe4c3ba4674ba94c...,3392,2026-09-11 20:48:35+00:00
4,nyc_gov.db/country_codes-9a177fe4c3ba4674ba94c...,4498,2026-09-11 20:48:35+00:00
...,...,...,...
933,test.db/my_table-873c67c261f94a94b2b8e8a31e676...,3708,2026-09-11 21:03:43+00:00
934,test.db/my_table-873c67c261f94a94b2b8e8a31e676...,975,2026-09-11 21:03:43+00:00
935,test.db/my_table-873c67c261f94a94b2b8e8a31e676...,7236,2026-09-11 21:03:43+00:00
936,test.db/my_table-873c67c261f94a94b2b8e8a31e676...,4488,2026-09-11 21:03:43+00:00
